# Análise visual: curvas de densidade (MoSS 70/30 vs datasets binários locais)

Este notebook faz o fluxo pedido:
1. Carrega `moss_binario_lite.pkl`;
2. Encontra a prevalência MoSS mais próxima de **70/30** (neg/pos = 0.70/0.30);
3. Usa os datasets binários locais em `datasets/binary`;
4. Gera batches 70/30 com o APP;
5. Compara visualmente as curvas de densidade dos scores reais vs MoSS.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
BATCH_SIZE = 100
N_PREV = 19
REPEATS = 30
TARGET_POS_PREV = 0.30

# Ajuste para o seu ambiente, se necessário
MOSS_PKL = '/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/moss/lite/moss_binario_lite.pkl'
DATASETS_ROOT = '/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/binary'

np.random.seed(SEED)

if not os.path.exists(MOSS_PKL):
    raise FileNotFoundError(f'MoSS PKL não encontrado: {MOSS_PKL}')
if not os.path.isdir(DATASETS_ROOT):
    raise FileNotFoundError(f'Pasta de datasets binários não encontrada: {DATASETS_ROOT}')

datasets = sorted([f for f in os.listdir(DATASETS_ROOT) if f.endswith('.csv')])
if not datasets:
    raise RuntimeError(f'Nenhum CSV encontrado em: {DATASETS_ROOT}')

print(f'Datasets binários encontrados: {len(datasets)}')


In [ ]:
# Implementação do APP (conforme solicitado)
def generate_prevalences(n_prev, repeats):
    prevalences = np.linspace(0.05, 0.95, n_prev)
    return [[1 - p, p] for p in prevalences] * repeats

def sample_batch(y, prevalence, batch_size):
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    n_pos = int(batch_size * prevalence[1])
    idx = np.concatenate([
        np.random.choice(pos, n_pos, replace=True),
        np.random.choice(neg, batch_size - n_pos, replace=True)
    ])
    np.random.shuffle(idx)
    return idx

class APP:
    def __init__(self):
        self.prevs = generate_prevalences(N_PREV, REPEATS)

    def split(self, X, y):
        for p in self.prevs:
            yield sample_batch(y, p, BATCH_SIZE), p

app = APP()
target_prev = [1 - TARGET_POS_PREV, TARGET_POS_PREV]
print('Prevalência alvo APP (neg,pos):', target_prev)
print('Existe no grid?', any(np.isclose(p[1], TARGET_POS_PREV) for p in app.prevs))


In [ ]:
def scalar_prev_from_moss_key(key):
    # chave possível: float ou tupla/lista com prevalência
    if isinstance(key, (int, float, np.floating)):
        return float(key)

    if isinstance(key, (tuple, list)):
        first = key[0]
        if isinstance(first, (int, float, np.floating)):
            return float(first)
        if isinstance(first, (tuple, list, np.ndarray)) and len(first) >= 2:
            return float(first[0])

    raise ValueError(f'Formato de chave MoSS não suportado: {key}')

def to_score_vector(curve):
    arr = np.asarray(curve)
    if arr.ndim == 1:
        return arr.astype(float)
    return arr[:, 0].astype(float)

with open(MOSS_PKL, 'rb') as f:
    moss_data = pickle.load(f)

moss_map = {}
for key, curves in moss_data.items():
    neg_prev = scalar_prev_from_moss_key(key)
    scores = np.concatenate([to_score_vector(c) for c in curves])
    moss_map[neg_prev] = scores

target_neg_prev = 0.70
chosen_neg_prev = min(moss_map.keys(), key=lambda p: abs(p - target_neg_prev))
moss_scores_7030 = moss_map[chosen_neg_prev]

print(f'Prevalência MoSS escolhida (neg): {chosen_neg_prev:.3f}')
print(f'Scores MoSS usados: {len(moss_scores_7030)}')


In [ ]:
def kde_curve(scores, grid):
    scores = np.asarray(scores, dtype=float)
    if len(scores) < 2 or np.allclose(scores.std(), 0):
        return np.zeros_like(grid)
    try:
        return gaussian_kde(scores)(grid)
    except Exception:
        return np.zeros_like(grid)

def load_local_binary_dataset(path):
    df = pd.read_csv(path)
    y = df.iloc[:, -1].values
    X = df.iloc[:, :-1].values

    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError(f'Dataset não binário: {path}')

    # garante rótulo positivo = 1
    y = (y == classes.max()).astype(int)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.5, stratify=y, random_state=SEED
    )

    scaler = StandardScaler().fit(Xtr)
    Xtr = scaler.transform(Xtr)
    Xte = scaler.transform(Xte)
    return Xtr, Xte, ytr, yte

def real_scores_app_7030(ds_csv):
    Xtr, Xte, ytr, yte = load_local_binary_dataset(ds_csv)

    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr)

    batches = []
    for idx, p in app.split(Xte, yte):
        if np.isclose(p[1], TARGET_POS_PREV):
            batches.append(clf.predict_proba(Xte[idx])[:, 1])

    if not batches:
        raise RuntimeError('APP não gerou batches para a prevalência alvo 70/30')

    return np.concatenate(batches)


In [ ]:
# Opcional: limitar número de datasets para análise rápida
MAX_DATASETS = 20  # coloque None para usar todos
selected = datasets[:MAX_DATASETS] if MAX_DATASETS is not None else datasets

grid = np.linspace(0, 1, 300)
moss_kde = kde_curve(moss_scores_7030, grid)

rows = []
for ds in selected:
    ds_path = os.path.join(DATASETS_ROOT, ds)
    try:
        real_scores = real_scores_app_7030(ds_path)
        real_kde = kde_curve(real_scores, grid)
        l1 = float(np.mean(np.abs(real_kde - moss_kde)))

        rows.append({
            'dataset': ds,
            'n_real_scores': len(real_scores),
            'n_moss_scores': len(moss_scores_7030),
            'kde_l1': l1,
            'real_scores': real_scores,
            'real_kde': real_kde
        })
    except Exception as e:
        print(f'[WARN] {ds}: {e}')

results = pd.DataFrame([{k: v for k, v in r.items() if k not in ['real_scores', 'real_kde']} for r in rows])
results = results.sort_values('kde_l1').reset_index(drop=True)
results.head(10)


In [ ]:
# Plot datasets mais próximos e mais distantes do MoSS (em termos de KDE L1)
TOPK = 4
if len(rows) == 0:
    raise RuntimeError('Nenhum dataset processado com sucesso')

rows_sorted = sorted(rows, key=lambda r: r['kde_l1'])
best = rows_sorted[:TOPK]
worst = rows_sorted[-TOPK:]

def plot_group(group, title):
    n = len(group)
    fig, axes = plt.subplots(n, 1, figsize=(8, 3*n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, row in zip(axes, group):
        ax.plot(grid, row['real_kde'], label=f"Real {row['dataset']}", linewidth=2)
        ax.plot(grid, moss_kde, label='MoSS 70/30', linewidth=2)
        ax.set_ylabel('Densidade')
        ax.set_title(f"{row['dataset']}  |  KDE-L1={row['kde_l1']:.4f}")
        ax.legend()

    axes[-1].set_xlabel('Score da classe positiva')
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

plot_group(best, 'Datasets mais próximos do MoSS (70/30)')
plot_group(worst, 'Datasets mais distantes do MoSS (70/30)')


In [ ]:
# Salvar ranking resumido
OUT_DIR = Path('results/exp_015_density_notebook')
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUT_DIR / 'density_comparison_summary.csv'
results.to_csv(summary_path, index=False)
print('Resumo salvo em:', summary_path)
